In [ ]:
# ============================================================================
# CELL 1: Configuration and Imports
# ============================================================================

import requests
import json
from typing import List, Dict, Any

# ===== CONFIGURATION - UPDATE THESE =====
VLLM_ENDPOINT = ""  # Your vLLM URL
VLLM_API_KEY = ""  # Your API key
MCP_ENDPOINT = "http://pg-airman-mcp-service.samouelian-dev.svc.cluster.local:8000/mcp"

# Model selection - change this to test different models
MODEL_NAME = "qwen3-14b"  # Options: "qwen3-14b" or "nvidia/nemotron-nano-9b-v2"
#MODEL_NAME = "nvidia-nemotron-nano-9b-v2"
# =========================================

print(f"Configuration loaded:")
print(f"  vLLM Endpoint: {VLLM_ENDPOINT}")
print(f"  MCP Endpoint: {MCP_ENDPOINT}")
print(f"  Model: {MODEL_NAME}")

Configuration loaded:
  vLLM Endpoint: https://litellm-prod.apps.maas.redhatworkshops.io/v1
  MCP Endpoint: http://pg-airman-mcp-service.samouelian-dev.svc.cluster.local:8000/mcp
  Model: qwen3-14b


In [2]:
# ============================================================================
# CELL 2: Helper Functions
# ============================================================================

def print_section(title: str):
    """Print a section header"""
    print("\n" + "="*80)
    print(f"  {title}")
    print("="*80)

def print_json(label: str, data: Dict):
    """Pretty print JSON with a label"""
    print(f"\n{label}:")
    print(json.dumps(data, indent=2))

def detect_model_format(model_name: str) -> str:
    """Detect tool calling format from model name"""
    if "nemotron" in model_name.lower():
        return "nemotron"
    else:
        return "openai"

def parse_sse_response(response_text: str) -> Dict:
    """
    Parse Server-Sent Events (SSE) response from MCP streamable-http transport.
    
    SSE format:
        data: {"jsonrpc": "2.0", "result": {...}}
        
        (blank line separates events)
    """
    lines = response_text.strip().split('\n')
    
    # Look for lines starting with "data: "
    for line in lines:
        line = line.strip()
        if line.startswith('data: '):
            json_str = line[6:]  # Remove 'data: ' prefix
            try:
                return json.loads(json_str)
            except json.JSONDecodeError as e:
                print(f"⚠️  Failed to parse JSON from SSE data: {e}")
                print(f"Raw data: {json_str[:200]}...")
                raise
    
    # If no SSE format found, response might be plain JSON or error
    print("⚠️  No SSE 'data:' lines found, attempting direct JSON parse")
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        print(f"Raw response text:\n{response_text}")
        raise

MODEL_FORMAT = detect_model_format(MODEL_NAME)
print(f"Detected tool calling format: {MODEL_FORMAT}")

Detected tool calling format: openai


In [3]:
# ============================================================================
# CELL 3: Discover MCP Tools (Session-Aware)
# ============================================================================

print_section("STEP 1: Discover MCP Tools")

# Headers required for MCP streamable-http transport
mcp_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream"
}

# Initialize MCP session
mcp_init_request = {
    "jsonrpc": "2.0",
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-11-25",
        "capabilities": {},
        "clientInfo": {
            "name": "jupyter-experiment",
            "version": "1.0.0"
        }
    },
    "id": 1
}

print_json("MCP Initialize Request", mcp_init_request)

# Send initialize request
response = requests.post(MCP_ENDPOINT, json=mcp_init_request, headers=mcp_headers)
print(f"\nResponse Status: {response.status_code}")
print(f"Response Headers: {dict(response.headers)}")

# Extract session ID from response headers
session_id = response.headers.get('mcp-session-id')
if session_id:
    print(f"\n✅ Session ID: {session_id}")
    # Add session ID to headers for all subsequent requests
    mcp_headers['mcp-session-id'] = session_id
else:
    print("\n⚠️  No session ID returned")

print(f"\nRaw Response Text:\n{response.text[:500]}...")

mcp_init_response = parse_sse_response(response.text)
print_json("MCP Initialize Response", mcp_init_response)

# List tools (with session ID in headers)
mcp_tools_request = {
    "jsonrpc": "2.0",
    "method": "tools/list",
    "params": {},
    "id": 2
}

print_json("MCP List Tools Request", mcp_tools_request)
print(f"Request Headers: {mcp_headers}")  # Show headers with session ID

response = requests.post(MCP_ENDPOINT, json=mcp_tools_request, headers=mcp_headers)
print(f"\nRaw Response Text:\n{response.text[:500]}...")

mcp_tools_response = parse_sse_response(response.text)
print_json("MCP List Tools Response", mcp_tools_response)

# Extract tools
mcp_tools = mcp_tools_response.get("result", {}).get("tools", [])
print(f"\n✅ Discovered {len(mcp_tools)} MCP tools")



  STEP 1: Discover MCP Tools

MCP Initialize Request:
{
  "jsonrpc": "2.0",
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "jupyter-experiment",
      "version": "1.0.0"
    }
  },
  "id": 1
}

Response Status: 200
Response Headers: {'date': 'Mon, 18 May 2026 21:16:08 GMT', 'server': 'uvicorn', 'cache-control': 'no-cache, no-transform', 'connection': 'keep-alive', 'content-type': 'text/event-stream', 'mcp-session-id': '40c9a43e7f094b7bb527e8254d22f64c', 'x-accel-buffering': 'no', 'Transfer-Encoding': 'chunked'}

✅ Session ID: 40c9a43e7f094b7bb527e8254d22f64c

Raw Response Text:
event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-11-25","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"pg-airman-mcp","version":"1.27.1"}}}

...

MCP Initialize Response:
{

In [4]:
# ============================================================================
# CELL 4: Convert MCP Tools to OpenAI Format
# ============================================================================

print_section("STEP 2: Convert MCP Tools to OpenAI Function Calling Format")

def convert_mcp_to_openai_tools(mcp_tools: List[Dict]) -> List[Dict]:
    """Convert MCP tool definitions to OpenAI function calling format"""
    openai_tools = []
    for tool in mcp_tools:
        openai_tool = {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool.get("description", ""),
                "parameters": tool.get("inputSchema", {
                    "type": "object",
                    "properties": {}
                })
            }
        }
        openai_tools.append(openai_tool)
    return openai_tools

openai_tools = convert_mcp_to_openai_tools(mcp_tools)
print_json("OpenAI-Format Tools", openai_tools[:2])  # Show first 2 for brevity
print(f"\n✅ Converted {len(openai_tools)} tools to OpenAI format")


  STEP 2: Convert MCP Tools to OpenAI Function Calling Format

OpenAI-Format Tools:
[
  {
    "type": "function",
    "function": {
      "name": "list_schemas",
      "description": "List all schemas in the database",
      "parameters": {
        "properties": {
          "noop": {
            "description": "Workaround parameter, always use 'doit'",
            "title": "Noop",
            "type": "string"
          }
        },
        "required": [
          "noop"
        ],
        "title": "list_schemasArguments",
        "type": "object"
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_objects",
      "description": "List objects in a schema with comments",
      "parameters": {
        "properties": {
          "schema_name": {
            "description": "Schema name",
            "title": "Schema Name",
            "type": "string"
          },
          "object_type": {
            "default": "table",
            "description": "Object t

In [8]:
# ============================================================================
# CELL 5: Build System Prompt and Messages
# ============================================================================

print_section("STEP 3: Build Messages Array")

system_prompt = """You are a PostgreSQL database analyst. You have access to tools for querying and analyzing the database.

When the user asks questions about the database, use the available tools to get information.
Be concise and direct in your responses."""

user_query = "List all tables in the public schema."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query}
]

print_json("Messages Array", messages)


  STEP 3: Build Messages Array

Messages Array:
[
  {
    "role": "system",
    "content": "You are a PostgreSQL database analyst. You have access to tools for querying and analyzing the database.\n\nWhen the user asks questions about the database, use the available tools to get information.\nBe concise and direct in your responses."
  },
  {
    "role": "user",
    "content": "List all tables in the public schema."
  }
]


In [9]:
# ============================================================================
# CELL 6: Send First Request to vLLM (Initial Query)
# ============================================================================

print_section(f"STEP 4: Send Query to vLLM ({MODEL_FORMAT} format)")

vllm_request = {
    "model": MODEL_NAME,
    "messages": messages,
    "tools": openai_tools,
    "tool_choice": "auto",
    "temperature": 0.1,
    "max_tokens": 2048,
    "stream": False  # Non-streaming for clarity
}

print_json("vLLM Request Payload", vllm_request)

# Send request to vLLM
headers = {
    "Authorization": f"Bearer {VLLM_API_KEY}",
    "Content-Type": "application/json"
}

response = requests.post(
    f"{VLLM_ENDPOINT}/chat/completions",
    headers=headers,
    json=vllm_request
)

vllm_response = response.json()
print_json("vLLM Response (Complete)", vllm_response)

# Extract the assistant's message
assistant_message = vllm_response["choices"][0]["message"]
print_json("Assistant Message", assistant_message)


  STEP 4: Send Query to vLLM (openai format)

vLLM Request Payload:
{
  "model": "qwen3-14b",
  "messages": [
    {
      "role": "system",
      "content": "You are a PostgreSQL database analyst. You have access to tools for querying and analyzing the database.\n\nWhen the user asks questions about the database, use the available tools to get information.\nBe concise and direct in your responses."
    },
    {
      "role": "user",
      "content": "List all tables in the public schema."
    }
  ],
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "list_schemas",
        "description": "List all schemas in the database",
        "parameters": {
          "properties": {
            "noop": {
              "description": "Workaround parameter, always use 'doit'",
              "title": "Noop",
              "type": "string"
            }
          },
          "required": [
            "noop"
          ],
          "title": "list_schemasArguments",
     

In [7]:
# ============================================================================
# CELL 7: Parse Tool Calls (Format-Specific)
# ============================================================================

print_section(f"STEP 5: Parse Tool Calls ({MODEL_FORMAT} format)")

def parse_openai_tool_calls(message: Dict) -> List[Dict]:
    """Parse tool calls from OpenAI-format response"""
    return message.get("tool_calls", [])

def parse_nemotron_tool_calls(content: str) -> List[Dict]:
    """Parse tool calls from Nemotron <TOOLCALL> tags"""
    import re
    import uuid
    
    if not content:
        return []
    
    # Match <TOOLCALL>...</TOOLCALL> tags
    pattern = r'<TOOLCALL>(.*?)</TOOLCALL>'
    matches = re.findall(pattern, content, re.DOTALL)
    
    tool_calls = []
    for match in matches:
        try:
            calls_data = json.loads(match.strip())
            if not isinstance(calls_data, list):
                calls_data = [calls_data]
            
            for call in calls_data:
                tool_call = {
                    "id": f"call_{uuid.uuid4().hex[:24]}",
                    "type": "function",
                    "function": {
                        "name": call["name"],
                        "arguments": json.dumps(call["arguments"])
                    }
                }
                tool_calls.append(tool_call)
        except json.JSONDecodeError as e:
            print(f"⚠️  Failed to parse tool call: {e}")
    
    return tool_calls

# Parse based on format
if MODEL_FORMAT == "openai":
    tool_calls = parse_openai_tool_calls(assistant_message)
    print("📌 OpenAI format: Tool calls extracted from 'tool_calls' field")
else:  # nemotron
    tool_calls = parse_nemotron_tool_calls(assistant_message.get("content", ""))
    print("📌 Nemotron format: Tool calls parsed from <TOOLCALL> tags in content")

print_json("Parsed Tool Calls", tool_calls)
print(f"\n✅ Found {len(tool_calls)} tool call(s)")


  STEP 5: Parse Tool Calls (openai format)
📌 OpenAI format: Tool calls extracted from 'tool_calls' field

Parsed Tool Calls:
[
  {
    "function": {
      "arguments": "{\"noop\": \"doit\"}",
      "name": "list_schemas"
    },
    "id": "chatcmpl-tool-c763a35e4a8342aabd4040ccf6f94cfd",
    "type": "function"
  }
]

✅ Found 1 tool call(s)


In [8]:
# ============================================================================
# CELL 8: Execute Tool via MCP
# ============================================================================

print_section("STEP 6: Execute Tool via MCP")

if not tool_calls:
    print("❌ No tool calls to execute")
else:
    tool_call = tool_calls[0]  # Execute first tool call
    tool_name = tool_call["function"]["name"]
    tool_arguments = json.loads(tool_call["function"]["arguments"])
    
    print(f"Executing tool: {tool_name}")
    print_json("Tool Arguments", tool_arguments)
    
    # Call MCP tool
    mcp_tool_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": tool_arguments
        },
        "id": 3
    }
    
    print_json("MCP Tool Call Request", mcp_tool_request)
    
    response = requests.post(MCP_ENDPOINT, json=mcp_tool_request, headers=mcp_headers)
    print(f"\nRaw Response Text:\n{response.text[:500]}...")  # Debug output
    
    mcp_tool_response = parse_sse_response(response.text)
    print_json("MCP Tool Call Response", mcp_tool_response)
    
    # Extract result
    tool_result = mcp_tool_response.get("result", {})
    tool_result_content = tool_result.get("content", [])
    
    # Get text from first content item
    if tool_result_content and len(tool_result_content) > 0:
        tool_result_text = tool_result_content[0].get("text", "")
        print(f"\n✅ Tool executed successfully")
        print(f"Result preview: {tool_result_text[:200]}...")
    else:
        tool_result_text = "No result"
        print("⚠️  Tool returned no content")


  STEP 6: Execute Tool via MCP
Executing tool: list_schemas

Tool Arguments:
{
  "noop": "doit"
}

MCP Tool Call Request:
{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "list_schemas",
    "arguments": {
      "noop": "doit"
    }
  },
  "id": 3
}

Raw Response Text:
event: message
data: {"jsonrpc":"2.0","id":3,"result":{"content":[{"type":"text","text":"[{'schema_name': 'information_schema', 'schema_owner': 'postgres', 'schema_type': 'System Information Schema'}, {'schema_name': 'pg_catalog', 'schema_owner': 'postgres', 'schema_type': 'System Schema'}, {'schema_name': 'pg_toast', 'schema_owner': 'postgres', 'schema_type': 'System Schema'}, {'schema_name': 'public', 'schema_owner': 'pg_database_owner', 'schema_type': 'User Schema'}]"}],"structuredContent":{...

MCP Tool Call Response:
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "[{'schema_name': 'information_schema', 'schema_owner': 'postgre

In [9]:
# ============================================================================
# CELL 9: Add Tool Result to Messages (Format-Specific)
# ============================================================================

print_section(f"STEP 7: Format Tool Result for vLLM ({MODEL_FORMAT} format)")

# Add assistant's message with tool call to conversation
if MODEL_FORMAT == "openai":
    # OpenAI format: include tool_calls field
    messages.append({
        "role": "assistant",
        "content": assistant_message.get("content", ""),
        "tool_calls": tool_calls
    })
    print("📌 OpenAI format: Added assistant message with 'tool_calls' field")
else:  # nemotron
    # Nemotron format: content includes <TOOLCALL> tags
    messages.append({
        "role": "assistant",
        "content": assistant_message.get("content", "")
    })
    print("📌 Nemotron format: Added assistant message with <TOOLCALL> in content")

# Add tool result
messages.append({
    "role": "tool",
    "tool_call_id": tool_call["id"],
    "content": tool_result_text
})

print_json("Updated Messages Array", messages)



  STEP 7: Format Tool Result for vLLM (nemotron format)
📌 Nemotron format: Added assistant message with <TOOLCALL> in content

Updated Messages Array:
[
  {
    "role": "system",
    "content": "You are a PostgreSQL database analyst. You have access to tools for querying and analyzing the database.\n\nWhen the user asks questions about the database, use the available tools to get information.\nBe concise and direct in your responses."
  },
  {
    "role": "user",
    "content": "List all schemas in the database"
  },
  {
    "role": "assistant",
    "content": "Okay, the user wants to list all schemas in the database. Let me check the available tools. There's a function called list_schemas. The parameters for list_schemas require a 'noop' which is a workaround and should always be 'doit'. The other parameters aren't required. So I need to call list_schemas with noop set to 'doit'. That should retrieve all the schemas. I don't see any other parameters needed here. Let me make sure I'm 

In [10]:
# ============================================================================
# CELL 10: Send Second Request to vLLM (With Tool Results)
# ============================================================================

print_section("STEP 8: Send Tool Results Back to vLLM")

vllm_request_2 = {
    "model": MODEL_NAME,
    "messages": messages,
    "tools": openai_tools,
    "tool_choice": "auto",
    "temperature": 0.1,
    "max_tokens": 2048,
    "stream": False
}

print_json("vLLM Request Payload (with tool results)", vllm_request_2)

response = requests.post(
    f"{VLLM_ENDPOINT}/chat/completions",
    headers=headers,
    json=vllm_request_2
)

vllm_response_2 = response.json()
print_json("vLLM Response (Complete)", vllm_response_2)

final_message = vllm_response_2["choices"][0]["message"]
print_json("Final Assistant Message", final_message)


  STEP 8: Send Tool Results Back to vLLM

vLLM Request Payload (with tool results):
{
  "model": "nvidia-nemotron-nano-9b-v2",
  "messages": [
    {
      "role": "system",
      "content": "You are a PostgreSQL database analyst. You have access to tools for querying and analyzing the database.\n\nWhen the user asks questions about the database, use the available tools to get information.\nBe concise and direct in your responses."
    },
    {
      "role": "user",
      "content": "List all schemas in the database"
    },
    {
      "role": "assistant",
      "content": "Okay, the user wants to list all schemas in the database. Let me check the available tools. There's a function called list_schemas. The parameters for list_schemas require a 'noop' which is a workaround and should always be 'doit'. The other parameters aren't required. So I need to call list_schemas with noop set to 'doit'. That should retrieve all the schemas. I don't see any other parameters needed here. Let me ma

In [11]:
# ============================================================================
# CELL 11: Display Final Response
# ============================================================================

print_section("STEP 9: Final Response")

final_content = final_message.get("content", "")

# Clean Nemotron artifacts if present
if MODEL_FORMAT == "nemotron":
    import re
    # Remove <TOOLCALL> tags from final response
    final_content = re.sub(r'<TOOLCALL>.*?</TOOLCALL>', '', final_content, flags=re.DOTALL)
    # Remove <think> tags if present
    final_content = re.sub(r'<think>.*?</think>', '', final_content, flags=re.DOTALL)
    final_content = final_content.strip()
    print("📌 Cleaned Nemotron-specific tags from response")

print("\n" + "="*80)
print("FINAL ANSWER:")
print("="*80)
print(final_content)
print("="*80)

print(f"\n✅ Experiment complete!")
print(f"\nModel Format: {MODEL_FORMAT}")
print(f"Tool Calls Made: {len(tool_calls)}")
print(f"Final Response Length: {len(final_content)} characters")


  STEP 9: Final Response
📌 Cleaned Nemotron-specific tags from response

FINAL ANSWER:
Okay, the user asked to list all schemas in the database. I called the list_schemas function with the required 'noop' parameter set to 'doit'. The response came back with four schemas: information_schema, pg_catalog, pg_toast, and public. 

Now I need to present this information clearly. Let me check each schema's details. The first three are system schemas owned by postgres, and the last one is a user schema owned by pg_database_owner. The user might be interested in knowing which are system vs user schemas.

I should format the response in a list, maybe bullet points, showing each schema name, owner, and type. That way the user can easily see the structure. Also, mention that these are the schemas available. Keep it concise as per the user's request. No need for extra details unless the user asks for more. Alright, time to put it all together.
</think>

Here are the schemas in the database:

- **i

In [ ]:
# ============================================================================
# CELL 12: Compare Formats (Educational Summary)
# ============================================================================

print_section("FORMAT COMPARISON SUMMARY")

print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                    OpenAI Format vs Nemotron Format                        ║
╚════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────┐
│ OPENAI FORMAT (Qwen3, Llama 3.1, etc.)                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│ Assistant Response:                                                         │
│   {                                                                          │
│     "role": "assistant",                                                    │
│     "content": "",                                                          │
│     "tool_calls": [                                                         │
│       {                                                                      │
│         "id": "call_abc123",                                                │
│         "type": "function",                                                 │
│         "function": {                                                       │
│           "name": "list_schemas",                                           │
│           "arguments": "{}"                                                 │
│         }                                                                    │
│       }                                                                      │
│     ]                                                                        │
│   }                                                                          │
│                                                                              │
│ Tool Result Message:                                                        │
│   {                                                                          │
│     "role": "tool",                                                         │
│     "tool_call_id": "call_abc123",                                          │
│     "content": "[schema results...]"                                        │
│   }                                                                          │
│                                                                              │
│ ✅ Structured, vLLM parses automatically with guided decoding               │
│ ✅ Standard OpenAI API format                                               │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│ NEMOTRON FORMAT (NVIDIA Nemotron models)                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│ Assistant Response:                                                         │
│   {                                                                          │
│     "role": "assistant",                                                    │
│     "content": "I'll check the schemas.                                    │
│                 <TOOLCALL>[{                                                │
│                   \\"name\\": \\"list_schemas\\",                          │
│                   \\"arguments\\": {}                                       │
│                 }]</TOOLCALL>"                                              │
│   }                                                                          │
│                                                                              │
│ Tool Result Message:                                                        │
│   {                                                                          │
│     "role": "tool",                                                         │
│     "tool_call_id": "call_xyz789",  # Client-generated                     │
│     "content": "[schema results...]"                                        │
│   }                                                                          │
│                                                                              │
│ ⚠️  Custom XML-like tags in content                                         │
│ ⚠️  Requires client-side regex parsing                                      │
│ ⚠️  vLLM tool-call-parser may not work correctly                            │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘

KEY DIFFERENCES:
  • OpenAI: Structured 'tool_calls' field (parsed by vLLM)
  • Nemotron: Tags embedded in 'content' field (requires regex parsing)
  • OpenAI: ID generated by vLLM
  • Nemotron: ID must be generated client-side
  • OpenAI: Works with vLLM's built-in guided decoding
  • Nemotron: Needs custom client parsing or correct vLLM parser config
  
CHAT TEMPLATE ROLE:
  • Both use Jinja2 templates to format messages → prompt text
  • Qwen3: <|im_start|>role\\ncontent<|im_end|>
  • Nemotron: Similar structure, model trained to output <TOOLCALL> tags
  • vLLM applies template before inference, removes template after generation
  
TOOL CALL PARSER ROLE:
  • OpenAI models: vLLM uses guided decoding, no parser needed
  • Nemotron: Parser should convert <TOOLCALL> → structured tool_calls
  • If parser misconfigured: client gets raw tags (current issue)
""")
